# HiveMind Benchmark Autopilot (Phase 1: 4B → Phase 2: 35B-MTP, je 3 Runs, FUSE-gehärtet)

Dieses Notebook führt deine Benchmark-Kampagne in **kompakten Einzel-Zellen pro Durchlauf (`Run 1/3` .. `Run 3/3`)** aus — zuerst **4B**, dann **35B-MTP** für direkten 1:1-Vergleich auf derselben A100/L4:

### Phase 1 — 4B Campaign (`qwen3.5:4b-mtp`)
- **Zellen `3a`–`3c` — 4B HiveMind Core (23 Tasks, 100k Ctx, Runs 1–3):** `run1_log1_core_run1.log` .. `run1_log3_core_run3.log` *(bereits fertig? Kann auf Account 2 einfach übersprungen werden).*
- **Zellen `4a`–`4c` — 4B Aider Polyglot (34 Tasks, 100k Ctx, Runs 1–3):** `run1_log4_polyglot_run1.log` .. `run1_log6_polyglot_run3.log` *(Mirror: `run1_log4_polyglot.log`; optional `4d`–`4e` Half-Slices).*
- **Zellen `5a`–`5c` — 4B DeepSWE (3 Repo-Tasks: `tomlkit`, `httpx`, `cattrs`, 128k Ctx auf A100 / 100k auf L4, Runs 1–3):** `run1_log7_deepswe_run1.log` .. `run1_log9_deepswe_run3.log` *(Mirror: `run1_log5_deepswe.log`).*

### Phase 2 — 35B-MTP Campaign (`hermes3.6:35b-a3b-uncensored-genesis-apex-mini`)
- **Zellen `6a`–`6c` — 35B-MTP Aider Polyglot (34 Tasks, 120k Ctx, Runs 1–3):** `35b_run1_log4_polyglot_run1.log` .. `35b_run1_log6_polyglot_run3.log` *(Mirror: `35b_run1_log4_polyglot.log`; optional `6d`–`6e` Half-Slices).*
- **Zellen `7a`–`7c` — 35B-MTP DeepSWE (3 Repo-Tasks, 120k/128k Ctx, Runs 1–3):** `35b_run1_log7_deepswe_run1.log` .. `35b_run1_log9_deepswe_run3.log` *(Mirror: `35b_run1_log5_deepswe.log`).*
- **Zellen `8a`–`8c` (Optional) — 35B-MTP HiveMind Core (23 Tasks, 120k Ctx, Runs 1–3):** `35b_run1_log1_core_run1.log` .. `35b_run1_log3_core_run3.log`.

### Auswertung, Overnight-Schutz & Harnesses
- **Zelle `1b` (AFK- & Deadlock-Inspektor):** Liest `STATUS_OVERVIEW.txt`, Log-Alter (`[WARN: >15m ...]`) und Log-Tails — ohne GPU/Engine zu starten.
- **Overnight Drive-FUSE-Resilience:** Log-Handles sind pro Handle mit `except OSError` geschützt (ein FUSE-Hickup killt keinen Run); `STATUS_OVERVIEW.txt` liest primär lokal und wird während des Streams auf max. `1x/10s` gedrosselt.
- **Anti-Hang-Fix:** Nach `wrote ...json` + `[bench] Saved ` beendet `run_stage()` die Stream-Schleife nach max. 5s, während die `setsid`-Engine für den nächsten Run warm im VRAM bleibt.
- **Zelle `9` & `10`:** Gesamt-Leaderboard (`_leaderboard_summary.json`) & Server-Stop (`rent_bootstrap.sh stop`).

In [ ]:
# 1) Google Drive mounten & vorhandene Checkpoints prüfen
import json
from collections import Counter
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hivemind_bench")
DRIVE_RESULTS = DRIVE_ROOT / "results"
DRIVE_LOGS = DRIVE_ROOT / "logs"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive verbunden: {DRIVE_RESULTS}")
except ImportError:
    DRIVE_RESULTS = Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
    print(f"Hinweis: google.colab nicht aktiv, lokaler Backup-Ordner: {DRIVE_RESULTS}")

existing = [p for p in sorted(DRIVE_RESULTS.glob("*.json")) if not p.name.startswith("_")]
if existing:
    print(f"\nGefundene Checkpoints in Drive ({len(existing)}):")
    for p in existing:
        try:
            d = json.loads(p.read_text(encoding="utf-8"))
            res = d.get("results", [])
            by_run = dict(sorted(Counter(int(r.get("run") or 1) for r in res).items()))
            print(f"  - {p.name:30s} | model={d.get('model')} ctx={d.get('num_ctx', '?')} rows={len(res)} runs={by_run}")
        except Exception as exc:
            print(f"  - {p.name:30s} | (nicht lesbar: {exc})")
else:
    print("\nNoch keine Checkpoints in Drive — frischer Start.")

In [ ]:
# 1b) AFK-Status & Deadlock/Log-Inspektor (schnell, startet KEINE GPU/Engine)
import time as _time_afk
from pathlib import Path as _Path_afk

if "DRIVE_LOGS" not in globals():
    _dr_afk = _Path_afk("/content/drive/MyDrive/hivemind_bench")
    DRIVE_RESULTS = (_dr_afk / "results") if _Path_afk("/content/drive/MyDrive").exists() else _Path_afk("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = (_dr_afk / "logs") if _Path_afk("/content/drive/MyDrive").exists() else _Path_afk("/root/HiveMind_llama/bench/logs_backup")

_ov_file = DRIVE_LOGS / "STATUS_OVERVIEW.txt"
if _ov_file.is_file():
    print(_ov_file.read_text(encoding="utf-8", errors="replace"))
else:
    print("[AFK] Noch keine STATUS_OVERVIEW.txt in Drive vorhanden.")

_4b_logs = [
    "run1_log1_core_run1.log",
    "run1_log2_core_run2.log",
    "run1_log3_core_run3.log",
    "run1_log4_polyglot_run1.log",
    "run1_log5_polyglot_run2.log",
    "run1_log6_polyglot_run3.log",
    "run1_log4_polyglot_part1.log",
    "run1_log4_polyglot_part2.log",
    "run1_log4_polyglot.log",
    "run1_log7_deepswe_run1.log",
    "run1_log8_deepswe_run2.log",
    "run1_log9_deepswe_run3.log",
    "run1_log5_deepswe.log",
]
_35b_logs = [
    "35b_run1_log4_polyglot_run1.log",
    "35b_run1_log5_polyglot_run2.log",
    "35b_run1_log6_polyglot_run3.log",
    "35b_run1_log4_polyglot_part1.log",
    "35b_run1_log4_polyglot_part2.log",
    "35b_run1_log4_polyglot.log",
    "35b_run1_log7_deepswe_run1.log",
    "35b_run1_log8_deepswe_run2.log",
    "35b_run1_log9_deepswe_run3.log",
    "35b_run1_log5_deepswe.log",
    "35b_run1_log1_core_run1.log",
    "35b_run1_log2_core_run2.log",
    "35b_run1_log3_core_run3.log",
]

_log_candidates = [p for p in DRIVE_LOGS.glob("*.log") if p.is_file()]
if _log_candidates:
    _newest = max(_log_candidates, key=lambda p: p.stat().st_mtime)
    _age_s = max(0, int(_time_afk.time() - _newest.stat().st_mtime))
    print(f"\n[AFK] Letzte Log-Aktivität: {_newest.name} vor {_age_s}s ({_age_s / 60.0:.1f} min)")
    if _age_s > 900:
        print("[AFK][WARN] Letzter Log-Eintrag älter als 15 Min. — prüfe, ob die aktive Zelle hängt oder beendet ist.")

print("\n=== Vorhandene Log-Dateien in Drive ===")
_shown = 0
for _ln in [*_4b_logs, *_35b_logs]:
    _lp = DRIVE_LOGS / _ln
    if _lp.is_file():
        _shown += 1
        _lines = _lp.read_text(encoding="utf-8", errors="replace").splitlines()
        _age_m = max(0.0, (_time_afk.time() - _lp.stat().st_mtime) / 60.0)
        print(f"\n--- {_ln} ({len(_lines)} Zeilen, {_lp.stat().st_size / 1024:.1f} KB, age={_age_m:.1f}m) [letzte 12 Zeilen] ---")
        for _tail in _lines[-12:]:
            print("  " + _tail)
if _shown == 0:
    print("  (noch keine Log-Dateien vorhanden)")

In [ ]:
# 2) Repo aktualisieren, Drive -> Lokal vorab syncen & Basis-Setup (inkl. Polyglot & DeepSWE)
import json, shutil, subprocess, sys
from pathlib import Path

if "DRIVE_RESULTS" not in globals():
    DRIVE_ROOT = Path("/content/drive/MyDrive/hivemind_bench")
    DRIVE_RESULTS = DRIVE_ROOT / "results" if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = DRIVE_ROOT / "logs" if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)

def _run_stream(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    assert p.stdout is not None
    for line in p.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed (exit={rc}): {cmd}")

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /root | tail -1

_run_stream(["bash", "-lc", (
    "[ -d /root/HiveMind_llama/.git ] || git clone --branch bench-stable https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama; "
    "cd /root/HiveMind_llama && git fetch origin bench-stable && git checkout bench-stable && git reset --hard origin/bench-stable && "
    "ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh && "
    "chmod +x /root/rent_bootstrap.sh && git log -1 --format='[bench] commit: %h %s'"
)])

# Vorher-Sync: Alle Drive-Checkpoints in das lokale results/-Verzeichnis ziehen
local_res = Path("/root/HiveMind_llama/bench/results")
local_logs = local_res / "logs"
local_res.mkdir(parents=True, exist_ok=True)
local_logs.mkdir(parents=True, exist_ok=True)

def _json_rows(p):
    try:
        return len(json.loads(Path(p).read_text(encoding="utf-8")).get("results", []))
    except Exception:
        return -1

for src in sorted(DRIVE_RESULTS.glob("*.json")):
    if src.name.startswith("_"):
        continue
    dst = local_res / src.name
    src_rows = _json_rows(src)
    dst_rows = _json_rows(dst) if dst.exists() else -1
    if (not dst.exists()) or (src_rows >= 0 and src_rows >= dst_rows):
        shutil.copy2(src, dst)
        print(f"[pre-sync] {src.name} ({max(0, src_rows)} Rows) -> {dst}")

# Sync bestehender Logs aus Drive -> lokal (damit Resumes nahtlos anknüpfen)
for src_log in sorted(DRIVE_LOGS.glob("*.log")):
    dst_log = local_logs / src_log.name
    if (not dst_log.exists()) or src_log.stat().st_size >= dst_log.stat().st_size:
        shutil.copy2(src_log, dst_log)

# Basis-Setup (Python 3.14, llama.cpp, Aider, Hermes, Pi + Node 22)
_run_stream(["bash", "-lc", "HM_MODEL=skip HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup"])

# Alle 3 Task-Corpora bereithalten (Core ist im Repo, Polyglot & DeepSWE werden importiert)
_poly_dir = Path("/root/HiveMind_llama/bench/tasks_polyglot")
_poly_count = len(list(_poly_dir.glob("polyglot_*/task.json")))
if _poly_count < 34:
    print(f"[setup] Polyglot nur {_poly_count}/34 Tasks vorhanden — importiere vollständigen Corpus.")
    _run_stream(["bash", "-lc", "HM_POLYGLOT_NAMES=all bash /root/rent_bootstrap.sh polyglot"])
else:
    print("[setup] Polyglot Tasks (34) bereits vollständig.")

try:
    _run_stream(["bash", "-lc", "cd /root/HiveMind_llama && uv pip install --python .venv/bin/python -q hypothesis trio attrs pytest pytest-asyncio anyio"])
except RuntimeError as _dep_exc:
    print(f"[setup][warn] DeepSWE test deps not installed: {_dep_exc}")
_ds_dir = Path("/root/HiveMind_llama/bench/tasks_deepswe")
_ds_count = len(list(_ds_dir.glob("*/task.json")))
if _ds_count < 3:
    print(f"[setup] DeepSWE nur {_ds_count}/3 Tasks vorhanden — importiere Corpus.")
    _run_stream(["bash", "/root/rent_bootstrap.sh", "deepswe"])
else:
    print("[setup] DeepSWE Tasks (3) bereits vollständig.")
print(f"[setup] BEREIT: results={DRIVE_RESULTS} | logs={DRIVE_LOGS}")

In [ ]:
# 2b) OPTIONAL: Alte Checkpoints und/oder Logs gezielt bereinigen
DELETE_EXISTING_BENCHMARKS = False  # True => löscht Checkpoints (.json) + Logs + Engine-State für kompletten Neustart
CLEAN_LOGS_ONLY = False             # True => löscht NUR alte *.log + STATUS_OVERVIEW.txt (behält JSON-Checkpoints)

import os as _reset_os, pathlib as _reset_pl, shutil as _reset_shutil, signal as _reset_sig, subprocess as _reset_subproc

_log_dirs = [
    _reset_pl.Path("/root/HiveMind_llama/bench/results/logs"),
    _reset_pl.Path("/root/HiveMind_llama/bench/logs_backup"),
    _reset_pl.Path("/root/HiveMind_llama/logs"),
    _reset_pl.Path("/content/drive/MyDrive/hivemind_bench/logs"),
]
_ckpt_dirs = [
    _reset_pl.Path("/root/HiveMind_llama/bench/results"),
    _reset_pl.Path("/root/HiveMind_llama/bench/results/external"),
    _reset_pl.Path("/root/HiveMind_llama/bench/results_backup"),
    _reset_pl.Path("/content/drive/MyDrive/hivemind_bench/results"),
    _reset_pl.Path("/content/drive/MyDrive/hivemind_bench/results/external"),
]
_log_files = [f for d in _log_dirs if d.exists() for f in (*d.glob("*.log"), d / "STATUS_OVERVIEW.txt") if f.exists()]
for _sf in (
    _reset_pl.Path("/root/HiveMind_llama/bench_full.log"),
    _reset_pl.Path("/root/HiveMind_llama/bench_engine.log"),
):
    if _sf.exists():
        _log_files.append(_sf)

_ckpt_files = [f for d in _ckpt_dirs if d.exists() for f in (*d.glob("*.json"), *d.glob("*.log"))]
for _sf in (
    _reset_pl.Path("/root/HiveMind_llama/bench_gpu_settings.json"),
    _reset_pl.Path("/root/HiveMind_llama/settings.json"),
    _reset_pl.Path("/root/HiveMind_llama/runtime_models.json"),
    _reset_pl.Path("/root/HiveMind_llama/memory.json"),
    _reset_pl.Path("/root/HiveMind_llama/token_stats.json"),
    _reset_pl.Path("/root/HiveMind_llama/run_counter.json"),
    _reset_pl.Path("/root/HiveMind_llama/.active_model"),
    _reset_pl.Path("/root/HiveMind_llama/.active_ctx"),
):
    if _sf.exists():
        _ckpt_files.append(_sf)
_work_dir = _reset_pl.Path("/root/HiveMind_llama/bench/.work")

if DELETE_EXISTING_BENCHMARKS:
    if _reset_pl.Path("/root/rent_bootstrap.sh").is_file():
        _reset_subproc.run(["bash", "/root/rent_bootstrap.sh", "stop"], check=False)
    _pid_f = _reset_pl.Path("/root/HiveMind_llama/bench_engine.pid")
    if _pid_f.exists():
        try:
            _reset_os.kill(int(_pid_f.read_text().strip()), _reset_sig.SIGTERM)
        except (OSError, ValueError):
            pass
        _pid_f.unlink(missing_ok=True)
    for _f in [*_log_files, *_ckpt_files]:
        _f.unlink(missing_ok=True)
    if _work_dir.exists():
        _reset_shutil.rmtree(_work_dir, ignore_errors=True)
    print(f"[reset] {len(_log_files) + len(_ckpt_files)} Benchmark-Dateien & Logs gelöscht — kompletter Neustart bereit.")
elif CLEAN_LOGS_ONLY:
    for _f in _log_files:
        _f.unlink(missing_ok=True)
    print(f"[reset] {len(_log_files)} Log-Dateien bereinigt (JSON-Checkpoints behalten).")
else:
    print(f"[reset] ÜBERSPRUNGEN ({len(_ckpt_files)} Checkpoints, {len(_log_files)} Logs behalten).")

In [ ]:
# 3) Helper laden (clean_old_bench_logs, update_status_overview, run_stage, run_autopilot)
import json as _json, os, shutil as _shutil, statistics as _stats, time as _time
import subprocess as _subprocess
import sys as _sys
from datetime import datetime as _dt
from pathlib import Path

if "DRIVE_RESULTS" not in globals() or "DRIVE_LOGS" not in globals():
    _dr = Path("/content/drive/MyDrive/hivemind_bench")
    DRIVE_RESULTS = (_dr / "results") if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/results_backup")
    DRIVE_LOGS = (_dr / "logs") if Path("/content/drive/MyDrive").exists() else Path("/root/HiveMind_llama/bench/logs_backup")
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)

LOCAL_LOG_DIR = Path("/root/HiveMind_llama/bench/results/logs")
LOCAL_LOG_DIR.mkdir(parents=True, exist_ok=True)

CONDITIONS = "hivemind_duo,hivemind_duo_lean,aider,hermes,pi"
CTX_4B_SMALL_SUITES = 102400
CTX_4B_DEEPSWE = 131072
CTX_35B_SMALL_SUITES = 122880
CTX_35B_DEEPSWE = 122880
SUITES = {
    "core": "bench/tasks",
    "polyglot": "bench/tasks_polyglot",
    "deepswe": "bench/tasks_deepswe",
}


def clean_old_bench_logs(clear_drive=True, include_status=True):
    """Entfernt alte Bench-Logs vor einem frischen 4B Core Run 1 (behält JSON-Checkpoints)."""
    removed = 0
    dirs = [LOCAL_LOG_DIR, Path("/root/HiveMind_llama/bench/logs_backup")]
    if clear_drive:
        dirs.append(DRIVE_LOGS)
    for d in dirs:
        if not d.exists():
            continue
        for f in d.glob("*.log"):
            try:
                f.unlink(missing_ok=True)
                removed += 1
            except OSError:
                pass
        if include_status:
            _st = d / "STATUS_OVERVIEW.txt"
            if _st.exists():
                try:
                    _st.unlink(missing_ok=True)
                    removed += 1
                except OSError:
                    pass
    print(f"[logs] {removed} alte Log-/Status-Dateien vor frischem Start bereinigt.", flush=True)


def _load_rows(json_path):
    p = Path(json_path)
    try:
        if not p.is_file():
            return []
        return _json.loads(p.read_text(encoding="utf-8")).get("results", [])
    except Exception:
        return []


def _row_count(p, run_filter=None):
    rows = _load_rows(p)
    if run_filter is not None:
        rows = [r for r in rows if int(r.get("run") or 1) == int(run_filter)]
    return len(rows)


def _detect_vram_gb():
    try:
        out = _subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            text=True,
        ).strip().splitlines()[0]
        return float(out) / 1024.0
    except Exception:
        return 15.0


def _resolve_model(model_choice="4b", ctx_small=None, ctx_deepswe=None, model_id=None, for_deepswe=False):
    vram_gb = _detect_vram_gb()
    use_model = str(model_choice).strip().lower()
    if use_model == "auto":
        use_model = "4b" if vram_gb < 20.0 else "35b-mtp"
    if use_model == "4b":
        resolved_model = model_id or "qwen3.5:4b-mtp"
        tag = "4b"
        if for_deepswe:
            ctx = int(ctx_small or CTX_4B_SMALL_SUITES) if vram_gb < 35.0 else int(ctx_deepswe or CTX_4B_DEEPSWE)
        else:
            ctx = int(ctx_small or CTX_4B_SMALL_SUITES)
    elif use_model == "35b-mtp":
        resolved_model = model_id or "hermes3.6:35b-a3b-uncensored-genesis-apex-mini"
        tag = "35b_mtp"
        if for_deepswe:
            ctx = int(ctx_deepswe or (131072 if vram_gb >= 60.0 else CTX_35B_DEEPSWE))
        else:
            ctx = int(ctx_small or CTX_35B_SMALL_SUITES)
    else:
        resolved_model = model_id or "qwen3.6:35b-a3b-ud"
        tag = "35b"
        if for_deepswe:
            ctx = int(ctx_deepswe or (131072 if vram_gb >= 60.0 else CTX_35B_SMALL_SUITES))
        else:
            ctx = int(ctx_small or CTX_35B_SMALL_SUITES)
    if for_deepswe:
        print(f"[stage] DeepSWE ({use_model}) on {vram_gb:.1f} GB VRAM -> ctx={ctx}.")
    return use_model, resolved_model, tag, ctx, vram_gb


def _default_log_names(suite_key, tag, run_no, task_slice=None):
    prefix = "" if tag == "4b" else ("35b_" if tag == "35b_mtp" else f"{tag}_")
    rno = max(1, int(run_no))
    if suite_key == "core":
        return f"{prefix}run1_log{rno}_core_run{rno}.log", None
    if suite_key == "polyglot":
        mirror = f"{prefix}run1_log4_polyglot.log"
        if task_slice is not None:
            part = 1 if int(task_slice[0]) == 0 else 2
            return f"{prefix}run1_log4_polyglot_part{part}.log", mirror
        return f"{prefix}run1_log{3 + rno}_polyglot_run{rno}.log", mirror
    return f"{prefix}run1_log{6 + rno}_deepswe_run{rno}.log", f"{prefix}run1_log5_deepswe.log"


def update_status_overview(active_stage="IDLE", active_log="", active_detail="", extra_note=""):
    """Schreibt atomar STATUS_OVERVIEW.txt (lokal-zuerst gelesen, Drive-Write FUSE-sicher)."""
    now = _dt.now().strftime("%Y-%m-%d %H:%M:%S")
    local_res = Path("/root/HiveMind_llama/bench/results")

    def _pick_rows(*fnames):
        # Local-first: vermeidet teure/blockierende Drive-FUSE-Reads während des aktiven Runs.
        best = []
        for fn in fnames:
            loc_rows = _load_rows(local_res / fn)
            if loc_rows:
                if len(loc_rows) > len(best):
                    best = loc_rows
                continue
            drv_rows = _load_rows(DRIVE_RESULTS / fn)
            if len(drv_rows) > len(best):
                best = drv_rows
        return best

    core_4b_rows = _pick_rows("core_4b_100k.json")
    poly_4b_rows = _pick_rows("polyglot_4b_100k.json")
    dswe_4b_rows = _pick_rows("deepswe_4b_128k.json", "deepswe_4b_100k.json")
    poly_35b_rows = _pick_rows("polyglot_35b_mtp_120k.json")
    dswe_35b_rows = _pick_rows("deepswe_35b_mtp_128k.json", "deepswe_35b_mtp_120k.json")
    core_35b_rows = _pick_rows("core_35b_mtp_120k.json")

    lines = [
        "=========================================================================",
        f"HIVEMIND BENCHMARK AUTOPILOT — STATUS OVERVIEW ({now})",
        "=========================================================================",
        f"Active stage : {active_stage}",
        f"Active log   : {active_log or '(none)'}",
    ]
    if active_detail:
        lines.append(f"Last update  : {active_detail}")
    if extra_note:
        lines.append(f"Note         : {extra_note}")

    def _safe_mtime(p):
        try:
            return p.stat().st_mtime
        except OSError:
            return 0.0

    # Bevorzuge lokales Log-Verzeichnis, Fallback auf Drive
    log_files = [p for p in LOCAL_LOG_DIR.glob("*.log") if p.is_file()] if LOCAL_LOG_DIR.exists() else []
    if not log_files and DRIVE_LOGS.exists():
        try:
            log_files = [p for p in DRIVE_LOGS.glob("*.log") if p.is_file()]
        except OSError:
            log_files = []
    if log_files:
        newest = max(log_files, key=_safe_mtime)
        mtime = _safe_mtime(newest)
        if mtime > 0:
            age_s = max(0, int(_time.time() - mtime))
            age_note = f"{age_s}s ago ({newest.name})"
            if age_s > 900 and not str(active_stage).startswith(("IDLE", "FINISHED", "SUMMARY")):
                age_note += " [WARN: >15m no log event — check if cell stalled]"
            lines.append(f"Log pulse    : {age_note}")

    cond_cnt = len([c for c in CONDITIONS.split(",") if c.strip()])
    next_cell = "All 4B & 35B-MTP runs finished — run Cell 9 (Leaderboard)."
    found_next = False

    def _emit_suite(suite_label, rows_list, task_total, specs, update_next=True):
        nonlocal next_cell, found_next
        target_cells = task_total * cond_cnt
        for rno, cell_id, lname in specs:
            rr = [r for r in rows_list if int(r.get("run") or 1) == rno]
            tasks_done = len({r.get("task") for r in rr if r.get("task")})
            state = "DONE" if len(rr) >= target_cells else ("IN_PROGRESS" if rr else "PENDING")
            if update_next and state != "DONE" and not found_next:
                next_cell = f"Cell {cell_id} ({lname})"
                found_next = True
            lines.append(
                f"  - [{cell_id}] {suite_label:20s} Run {rno}/3 [{state:11s}] "
                f"{len(rr):3d}/{target_cells:<3d} cells ({tasks_done:2d}/{task_total} tasks) | log: {lname}"
            )

    lines.append("")
    lines.append("Phase 1 — 4B Campaign (Core -> Aider Polyglot -> DeepSWE):")
    _emit_suite("4B Core", core_4b_rows, 23, [
        (1, "3a", "run1_log1_core_run1.log"),
        (2, "3b", "run1_log2_core_run2.log"),
        (3, "3c", "run1_log3_core_run3.log"),
    ])
    _emit_suite("4B Polyglot", poly_4b_rows, 34, [
        (1, "4a", "run1_log4_polyglot_run1.log"),
        (2, "4b", "run1_log5_polyglot_run2.log"),
        (3, "4c", "run1_log6_polyglot_run3.log"),
    ])
    _emit_suite("4B DeepSWE", dswe_4b_rows, 3, [
        (1, "5a", "run1_log7_deepswe_run1.log"),
        (2, "5b", "run1_log8_deepswe_run2.log"),
        (3, "5c", "run1_log9_deepswe_run3.log"),
    ])

    lines.append("")
    lines.append("Phase 2 — 35B-MTP Campaign (Aider Polyglot -> DeepSWE + Optional Core):")
    _emit_suite("35B-MTP Polyglot", poly_35b_rows, 34, [
        (1, "6a", "35b_run1_log4_polyglot_run1.log"),
        (2, "6b", "35b_run1_log5_polyglot_run2.log"),
        (3, "6c", "35b_run1_log6_polyglot_run3.log"),
    ])
    _emit_suite("35B-MTP DeepSWE", dswe_35b_rows, 3, [
        (1, "7a", "35b_run1_log7_deepswe_run1.log"),
        (2, "7b", "35b_run1_log8_deepswe_run2.log"),
        (3, "7c", "35b_run1_log9_deepswe_run3.log"),
    ])
    _emit_suite("35B-MTP Core (Opt)", core_35b_rows, 23, [
        (1, "8a", "35b_run1_log1_core_run1.log"),
        (2, "8b", "35b_run1_log2_core_run2.log"),
        (3, "8c", "35b_run1_log3_core_run3.log"),
    ], update_next=False)

    lines.append(f"\nNext / Resume Cell: {next_cell}")

    def _fmt_scoreboard(title, rlist):
        if not rlist:
            return
        lines.append(f"\n{title} ({len(rlist)} cells):")
        for cond_name in [c.strip() for c in CONDITIONS.split(",") if c.strip()]:
            cr = [r for r in rlist if r.get("condition") == cond_name]
            if not cr:
                continue
            ok_n = sum(1 for r in cr if r.get("ok"))
            pct = 100.0 * ok_n / len(cr)
            med_s = _stats.median([float(r.get("seconds") or 0.0) for r in cr])
            tps_vals = [float(r.get("tok_per_sec") or 0.0) for r in cr if float(r.get("tok_per_sec") or 0.0) > 0]
            med_tps = _stats.median(tps_vals) if tps_vals else 0.0
            lines.append(f"  {cond_name:20s}: {ok_n:3d}/{len(cr):<3d} PASS ({pct:5.1f}%) | median {med_s:6.1f}s | {med_tps:6.1f} tok/s")

    _fmt_scoreboard("4B Core Combined Scoreboard", core_4b_rows)
    _fmt_scoreboard("4B Aider Polyglot Combined Scoreboard", poly_4b_rows)
    _fmt_scoreboard("4B DeepSWE Combined Scoreboard", dswe_4b_rows)
    _fmt_scoreboard("35B-MTP Aider Polyglot Combined Scoreboard", poly_35b_rows)
    _fmt_scoreboard("35B-MTP DeepSWE Combined Scoreboard", dswe_35b_rows)
    _fmt_scoreboard("Optional 35B-MTP Core Scoreboard", core_35b_rows)

    text = "\n".join(lines) + "\n"
    for target_dir in (LOCAL_LOG_DIR, DRIVE_LOGS):
        try:
            target_dir.mkdir(parents=True, exist_ok=True)
            tmp_p = target_dir / "_STATUS_OVERVIEW.tmp"
            tmp_p.write_text(text, encoding="utf-8")
            tmp_p.replace(target_dir / "STATUS_OVERVIEW.txt")
        except OSError:
            pass
    return text


def run_stage(suite, model_choice="4b", run=1, conditions=CONDITIONS,
              ctx_small=None, ctx_deepswe=None, model_id=None, only=None, task_slice=None,
              cell_timeout=None, task_timeout=None, resume_failed=False,
              log_name=None, mirror_log=None, stage_label=None,
              clean_logs_if_fresh=True):
    """Führt genau EINEN Durchlauf/Slice aus, mit FUSE-resistenten Log-Writes & gedrosseltem STATUS-Update."""
    suite_key = str(suite).strip().lower()
    if suite_key not in SUITES:
        raise ValueError(f"Unbekannte Suite: {suite!r} (erlaubt: {list(SUITES)})")
    for_deepswe = (suite_key == "deepswe")
    use_model, resolved_model, tag, ctx_tokens, vram_gb = _resolve_model(
        model_choice, ctx_small=ctx_small, ctx_deepswe=ctx_deepswe, model_id=model_id, for_deepswe=for_deepswe
    )
    if use_model.startswith("35b") and vram_gb < 20.0:
        print(f"[stage] SKIP {suite_key} ({use_model}): {vram_gb:.1f} GB VRAM < 20 GB.")
        return

    cond_list = [c.strip() for c in conditions.split(",") if c.strip()]
    if ctx_tokens < 65536 and "hermes" in cond_list:
        cond_list.remove("hermes")
        print(f"[stage] Hermes omitted at ctx={ctx_tokens}: needs >=64k.")
    stage_conditions = ",".join(cond_list)
    if not stage_conditions:
        print(f"[stage] SKIP {suite_key}: keine gueltigen Conditions.")
        return

    tasks_rel = SUITES[suite_key]
    only_str = str(only or "").strip()
    slice_label = ""
    if task_slice is not None:
        tasks_dir = Path("/root/HiveMind_llama") / tasks_rel
        if suite_key in ("polyglot", "deepswe") and (not tasks_dir.is_dir() or not any(tasks_dir.iterdir())):
            print(f"[stage] Preparing {suite_key} tasks before slicing...", flush=True)
            _subprocess.run(["bash", "/root/rent_bootstrap.sh", suite_key], check=True)
        all_ids = sorted(
            p.name for p in tasks_dir.iterdir()
            if p.is_dir() and (p / "task.json").is_file()
        )
        start_i, end_i = task_slice
        selected = all_ids[start_i:end_i]
        if not selected:
            print(f"[stage] SKIP {suite_key}: leerer Task-Slice {task_slice} (gesamt {len(all_ids)}).")
            return
        only_str = ",".join(selected)
        slice_label = f" | slice {start_i + 1}-{start_i + len(selected)}/{len(all_ids)}"

    stage_name = f"{suite_key}_{tag}_{ctx_tokens // 1024}k"
    out_file = Path(f"/root/HiveMind_llama/bench/results/{stage_name}.json")
    _saved = DRIVE_RESULTS / out_file.name
    out_file.parent.mkdir(parents=True, exist_ok=True)
    if _saved.is_file():
        try:
            if _saved.resolve() != out_file.resolve() and _row_count(_saved) >= _row_count(out_file):
                _shutil.copy2(_saved, out_file)
                print(f"[pre-sync] Drive -> Lokal: {_saved.name} ({_row_count(_saved)} Rows)", flush=True)
        except OSError as _pre_exc:
            print(f"[warn] Pre-Sync fehlgeschlagen: {_pre_exc}", flush=True)

    run_no = max(1, int(run))
    def_log, def_mirror = _default_log_names(suite_key, tag, run_no, task_slice=task_slice)
    log_name = str(log_name or def_log)
    mirror_log = def_mirror if mirror_log is None else (str(mirror_log) or None)
    stage_label = str(stage_label or f"{suite_key} ({tag}) run {run_no}{slice_label}")

    if clean_logs_if_fresh and suite_key == "core" and tag == "4b" and run_no == 1 and _row_count(out_file, run_filter=1) == 0:
        clean_old_bench_logs(clear_drive=True, include_status=True)

    eff_cell_to = int(cell_timeout or (1800 if for_deepswe else 900))
    print("\n=================================================================")
    print(f"[STAGE] {stage_label} | {stage_name} | run={run_no}{slice_label} | log={log_name}")
    print(f"        model={resolved_model} | ctx={ctx_tokens} | conds={stage_conditions}")
    print("=================================================================", flush=True)

    env = os.environ.copy()
    env.update({
        "HM_MODEL": resolved_model,
        "HM_CTX": str(ctx_tokens),
        "HM_TASKS": tasks_rel,
        "HM_RUNS": "1",
        "HM_RUN_OFFSET": str(run_no - 1),
        "HM_CONDITIONS": stage_conditions,
        "HM_CELL_TIMEOUT": str(eff_cell_to),
        "HM_TASK_TIMEOUT": str(int(task_timeout)) if task_timeout else "",
        "HM_ONLY": only_str,
        "HM_OUT": str(out_file),
        "HM_SYNC_DIR": str(DRIVE_RESULTS),
        "HM_RESUME": "1",
        "HM_RESUME_FAILED": "1" if resume_failed else "0",
    })

    _open_logs = []
    _target_names = [log_name] + ([mirror_log] if mirror_log and mirror_log != log_name else [])
    for _d in (LOCAL_LOG_DIR, DRIVE_LOGS):
        try:
            _d.mkdir(parents=True, exist_ok=True)
            for _nm in _target_names:
                _fh = open(_d / _nm, "a", encoding="utf-8")
                _fh.write(f"\n=== STAGE {stage_label} ({stage_name} run={run_no}{slice_label}) START @ {_dt.now().isoformat(timespec='seconds')} ===\n")
                _fh.flush()
                _open_logs.append(_fh)
        except OSError as _log_exc:
            print(f"[warn] Logziel {_d} nicht verfügbar ({_log_exc})", flush=True)

    update_status_overview(active_stage=stage_label, active_log=log_name, active_detail="Starting stage ...")
    rc = 1
    _saw_wrote = False
    _saw_saved = False
    _last_status_ts = _time.monotonic()
    try:
        proc = _subprocess.Popen(
            ["bash", "/root/rent_bootstrap.sh", "fg"], env=env,
            stdout=_subprocess.PIPE, stderr=_subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        )
        try:
            assert proc.stdout is not None
            for line in proc.stdout:
                _sys.stdout.write(line)
                _sys.stdout.flush()
                for _fh in list(_open_logs):
                    try:
                        _fh.write(line)
                        _fh.flush()
                    except OSError:
                        _open_logs.remove(_fh)
                        try:
                            _fh.close()
                        except OSError:
                            pass
                s = line.strip()
                if (" PASS " in line) or (" FAIL " in line) or s.startswith("[progress]") or ("still running" in line):
                    _now_ts = _time.monotonic()
                    if (_now_ts - _last_status_ts) >= 10.0:
                        update_status_overview(
                            active_stage=stage_label,
                            active_log=log_name,
                            active_detail=s,
                        )
                        _last_status_ts = _now_ts
                if s.startswith("wrote ") and s.endswith(".json"):
                    _saw_wrote = True
                if s.startswith("[bench] Saved "):
                    _saw_saved = True
                    break
            try:
                proc.stdout.close()
            except Exception:
                pass
            try:
                rc = proc.wait(timeout=5)
            except _subprocess.TimeoutExpired:
                try:
                    proc.terminate()
                    rc = proc.wait(timeout=3)
                except Exception:
                    try:
                        proc.kill()
                    except Exception:
                        pass
                    rc = 0 if (_saw_wrote and _saw_saved) else 1
            if _saw_wrote and _saw_saved and rc != 0:
                rc = 0
        except KeyboardInterrupt:
            proc.terminate()
            proc.wait(timeout=10)
            raise
    finally:
        for _fh in _open_logs:
            try:
                _fh.write(f"=== STAGE {stage_label} END (exit={rc}) @ {_dt.now().isoformat(timespec='seconds')} ===\n")
                _fh.close()
            except OSError:
                pass
        _eng_log = Path("/root/HiveMind_llama/bench_engine.log")
        if _eng_log.exists():
            try:
                _shutil.copy2(_eng_log, DRIVE_LOGS / f"{stage_name}_engine.log")
            except OSError:
                pass
        if out_file.is_file():
            try:
                if _saved.resolve() != out_file.resolve():
                    _shutil.copy2(out_file, _saved)
            except OSError as _sync_exc:
                print(f"[warn] Final Drive-Sync fehlgeschlagen: {_sync_exc}", flush=True)

    _chk = out_file if out_file.exists() else (_saved if _saved.exists() else None)
    _rows = _load_rows(_chk) if _chk else []
    _run_rows = [r for r in _rows if int(r.get("run") or 1) == run_no]
    _ok = sum(1 for r in _rows if r.get("ok"))
    _run_ok = sum(1 for r in _run_rows if r.get("ok"))
    _sz = f"{_chk.stat().st_size / 1024:.1f} KB" if _chk else "0 KB"
    update_status_overview(
        active_stage=f"{'FINISHED' if rc == 0 else 'ERROR'}: {stage_label}",
        active_log=log_name,
        active_detail=f"Run {run_no}: {_run_ok}/{len(_run_rows)} PASS | Gesamt: {_ok}/{len(_rows)} PASS (exit={rc})",
    )
    if rc != 0:
        raise RuntimeError(f"Stage {stage_label} (run={run_no}) abgebrochen (exit={rc}). Gesichert: {len(_rows)} Rows ({_sz}) -> {_saved}")
    print(f"[SAVE OK] {out_file.name} | Log: {DRIVE_LOGS / log_name} | Run {run_no}: {len(_run_rows)} Rows ({_run_ok} PASS) | Gesamt: {len(_rows)} Rows ({_ok} PASS) ({_sz})", flush=True)


def run_autopilot(model_choice="4b", core_runs=3, polyglot_runs=1,
                  deepswe_runs=0, conditions=CONDITIONS,
                  ctx_small=None, model_id=None):
    """Kompatibilitäts-Wrapper: führt Core (pro Run), Polyglot & DeepSWE sequentiell aus."""
    for r in range(1, int(core_runs) + 1):
        run_stage("core", model_choice=model_choice, run=r, conditions=conditions, ctx_small=ctx_small, model_id=model_id)
    for r in range(1, int(polyglot_runs) + 1):
        run_stage("polyglot", model_choice=model_choice, run=r, conditions=conditions, ctx_small=ctx_small, model_id=model_id)
    for r in range(1, int(deepswe_runs) + 1):
        run_stage("deepswe", model_choice=model_choice, run=r, conditions=conditions, ctx_small=ctx_small, model_id=model_id)


print(f"[helper] Bereit. Results: {DRIVE_RESULTS} | Logs: {DRIVE_LOGS} | Status: {DRIVE_LOGS / 'STATUS_OVERVIEW.txt'}")

In [ ]:
# 3a) 4B Core (23 Tasks, 100k Ctx) — Run 1/3 -> Log: run1_log1_core_run1.log
run_stage("core", "4b", run=1, log_name="run1_log1_core_run1.log", stage_label="4B Core Run 1/3")

In [ ]:
# 3b) 4B Core (23 Tasks, 100k Ctx) — Run 2/3 -> Log: run1_log2_core_run2.log
run_stage("core", "4b", run=2, log_name="run1_log2_core_run2.log", stage_label="4B Core Run 2/3")

In [ ]:
# 3c) 4B Core (23 Tasks, 100k Ctx) — Run 3/3 -> Log: run1_log3_core_run3.log
run_stage("core", "4b", run=3, log_name="run1_log3_core_run3.log", stage_label="4B Core Run 3/3")

In [ ]:
# 4a) 4B Aider Polyglot (34 Tasks, 100k Ctx) — Run 1/3 -> Log: run1_log4_polyglot_run1.log (+ run1_log4_polyglot.log)
run_stage(
    "polyglot", "4b", run=1,
    log_name="run1_log4_polyglot_run1.log",
    mirror_log="run1_log4_polyglot.log",
    stage_label="4B Polyglot Run 1/3",
)

In [ ]:
# 4b) 4B Aider Polyglot (34 Tasks, 100k Ctx) — Run 2/3 -> Log: run1_log5_polyglot_run2.log (+ run1_log4_polyglot.log)
run_stage(
    "polyglot", "4b", run=2,
    log_name="run1_log5_polyglot_run2.log",
    mirror_log="run1_log4_polyglot.log",
    stage_label="4B Polyglot Run 2/3",
)

In [ ]:
# 4c) 4B Aider Polyglot (34 Tasks, 100k Ctx) — Run 3/3 -> Log: run1_log6_polyglot_run3.log (+ run1_log4_polyglot.log)
run_stage(
    "polyglot", "4b", run=3,
    log_name="run1_log6_polyglot_run3.log",
    mirror_log="run1_log4_polyglot.log",
    stage_label="4B Polyglot Run 3/3",
)

In [ ]:
# 4d) OPTIONAL 4B Polyglot Half-Slice — Teil 1/2 (Tasks 1–17)
run_stage(
    "polyglot", "4b", run=1, task_slice=(0, 17),
    log_name="run1_log4_polyglot_part1.log",
    mirror_log="run1_log4_polyglot.log",
    stage_label="4B Polyglot Part 1/2 (Tasks 1-17)",
)

In [ ]:
# 4e) OPTIONAL 4B Polyglot Half-Slice — Teil 2/2 (Tasks 18–34)
run_stage(
    "polyglot", "4b", run=1, task_slice=(17, 34),
    log_name="run1_log4_polyglot_part2.log",
    mirror_log="run1_log4_polyglot.log",
    stage_label="4B Polyglot Part 2/2 (Tasks 18-34)",
)

In [ ]:
# 5a) 4B DeepSWE (3 Repo-Tasks: tomlkit, httpx, cattrs, 128k A100 / 100k L4) — Run 1/3
run_stage(
    "deepswe", "4b", run=1,
    log_name="run1_log7_deepswe_run1.log",
    mirror_log="run1_log5_deepswe.log",
    stage_label="4B DeepSWE Run 1/3",
)

In [ ]:
# 5b) 4B DeepSWE (3 Repo-Tasks: tomlkit, httpx, cattrs, 128k A100 / 100k L4) — Run 2/3
run_stage(
    "deepswe", "4b", run=2,
    log_name="run1_log8_deepswe_run2.log",
    mirror_log="run1_log5_deepswe.log",
    stage_label="4B DeepSWE Run 2/3",
)

In [ ]:
# 5c) 4B DeepSWE (3 Repo-Tasks: tomlkit, httpx, cattrs, 128k A100 / 100k L4) — Run 3/3
run_stage(
    "deepswe", "4b", run=3,
    log_name="run1_log9_deepswe_run3.log",
    mirror_log="run1_log5_deepswe.log",
    stage_label="4B DeepSWE Run 3/3",
)

In [ ]:
# 6a) 35B-MTP Aider Polyglot (34 Tasks, 120k Ctx) — Run 1/3
run_stage(
    "polyglot", "35b-mtp", run=1, ctx_small=122880,
    log_name="35b_run1_log4_polyglot_run1.log",
    mirror_log="35b_run1_log4_polyglot.log",
    stage_label="35B-MTP Polyglot Run 1/3",
)

In [ ]:
# 6b) 35B-MTP Aider Polyglot (34 Tasks, 120k Ctx) — Run 2/3
run_stage(
    "polyglot", "35b-mtp", run=2, ctx_small=122880,
    log_name="35b_run1_log5_polyglot_run2.log",
    mirror_log="35b_run1_log4_polyglot.log",
    stage_label="35B-MTP Polyglot Run 2/3",
)

In [ ]:
# 6c) 35B-MTP Aider Polyglot (34 Tasks, 120k Ctx) — Run 3/3
run_stage(
    "polyglot", "35b-mtp", run=3, ctx_small=122880,
    log_name="35b_run1_log6_polyglot_run3.log",
    mirror_log="35b_run1_log4_polyglot.log",
    stage_label="35B-MTP Polyglot Run 3/3",
)

In [ ]:
# 6d) OPTIONAL 35B-MTP Polyglot Half-Slice — Teil 1/2 (Tasks 1–17)
run_stage(
    "polyglot", "35b-mtp", run=1, task_slice=(0, 17), ctx_small=122880,
    log_name="35b_run1_log4_polyglot_part1.log",
    mirror_log="35b_run1_log4_polyglot.log",
    stage_label="35B-MTP Polyglot Part 1/2 (Tasks 1-17)",
)

In [ ]:
# 6e) OPTIONAL 35B-MTP Polyglot Half-Slice — Teil 2/2 (Tasks 18–34)
run_stage(
    "polyglot", "35b-mtp", run=1, task_slice=(17, 34), ctx_small=122880,
    log_name="35b_run1_log4_polyglot_part2.log",
    mirror_log="35b_run1_log4_polyglot.log",
    stage_label="35B-MTP Polyglot Part 2/2 (Tasks 18-34)",
)

In [ ]:
# 7a) 35B-MTP DeepSWE (3 Repo-Tasks: tomlkit, httpx, cattrs, 120k/128k Ctx) — Run 1/3
run_stage(
    "deepswe", "35b-mtp", run=1,
    log_name="35b_run1_log7_deepswe_run1.log",
    mirror_log="35b_run1_log5_deepswe.log",
    stage_label="35B-MTP DeepSWE Run 1/3",
)

In [ ]:
# 7b) 35B-MTP DeepSWE (3 Repo-Tasks: tomlkit, httpx, cattrs, 120k/128k Ctx) — Run 2/3
run_stage(
    "deepswe", "35b-mtp", run=2,
    log_name="35b_run1_log8_deepswe_run2.log",
    mirror_log="35b_run1_log5_deepswe.log",
    stage_label="35B-MTP DeepSWE Run 2/3",
)

In [ ]:
# 7c) 35B-MTP DeepSWE (3 Repo-Tasks: tomlkit, httpx, cattrs, 120k/128k Ctx) — Run 3/3
run_stage(
    "deepswe", "35b-mtp", run=3,
    log_name="35b_run1_log9_deepswe_run3.log",
    mirror_log="35b_run1_log5_deepswe.log",
    stage_label="35B-MTP DeepSWE Run 3/3",
)

In [ ]:
# 8a) OPTIONAL 35B-MTP Core (23 Tasks, 120k Ctx) — Run 1/3
run_stage("core", "35b-mtp", run=1, ctx_small=122880, log_name="35b_run1_log1_core_run1.log", stage_label="35B-MTP Core Run 1/3")

In [ ]:
# 8b) OPTIONAL 35B-MTP Core (23 Tasks, 120k Ctx) — Run 2/3
run_stage("core", "35b-mtp", run=2, ctx_small=122880, log_name="35b_run1_log2_core_run2.log", stage_label="35B-MTP Core Run 2/3")

In [ ]:
# 8c) OPTIONAL 35B-MTP Core (23 Tasks, 120k Ctx) — Run 3/3
run_stage("core", "35b-mtp", run=3, ctx_small=122880, log_name="35b_run1_log3_core_run3.log", stage_label="35B-MTP Core Run 3/3")

In [ ]:
# 9) Leaderboard & Gesamt-Export über alle in Google Drive gespeicherten Benchmarks
import json as _json_lb, subprocess as _subprocess_lb, sys as _sys_lb
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive as _colab_drive
        _colab_drive.mount("/content/drive")
    except ImportError:
        pass

if "DRIVE_RESULTS" not in globals():
    _dr_lb = Path("/content/drive/MyDrive/hivemind_bench")
    _loc_lb = Path("/root/HiveMind_llama/bench/results")
    _bak_lb = Path("/root/HiveMind_llama/bench/results_backup")
    if Path("/content/drive/MyDrive").exists():
        DRIVE_RESULTS = _dr_lb / "results"
        DRIVE_LOGS = _dr_lb / "logs"
    elif _loc_lb.exists() and any(not p.name.startswith("_") for p in _loc_lb.glob("*.json")):
        DRIVE_RESULTS = _loc_lb
        DRIVE_LOGS = _loc_lb / "logs"
    else:
        DRIVE_RESULTS = _bak_lb
        DRIVE_LOGS = Path("/root/HiveMind_llama/bench/logs_backup")

if "update_status_overview" in globals():
    print(update_status_overview(active_stage="SUMMARY", active_log="", active_detail="Generated final leaderboard"))

_repo_lb = Path("/root/HiveMind_llama")
_analyze_lb = _repo_lb / "bench" / "analyze.py"
if not _analyze_lb.is_file():
    print("[leaderboard] bench/analyze.py nicht lokal vorhanden — klone bench-stable shallow...")
    if (_repo_lb / ".git").exists():
        _subprocess_lb.run(["git", "-C", str(_repo_lb), "fetch", "origin", "bench-stable"], check=False)
        _subprocess_lb.run(["git", "-C", str(_repo_lb), "checkout", "bench-stable"], check=False)
    else:
        _subprocess_lb.run(
            ["git", "clone", "--depth", "1", "--branch", "bench-stable",
             "https://github.com/BoredLuzo/HiveMind_llama.git", str(_repo_lb)],
            check=True,
        )

_venv_py = _repo_lb / ".venv" / "bin" / "python"
_py_lb = str(_venv_py) if _venv_py.is_file() else _sys_lb.executable

files = [p for p in sorted(DRIVE_RESULTS.glob("*.json")) if not p.name.startswith("_")] if DRIVE_RESULTS.exists() else []
if not files:
    print(f"Noch keine JSON-Ergebnisse in {DRIVE_RESULTS} gefunden.")
else:
    _combined_export = {"source_dir": str(DRIVE_RESULTS), "per_file": {}, "combined": None}
    _valid_files = []
    for f in files:
        print(f"\n==================== {f.name} ====================")
        try:
            _data = _json_lb.loads(f.read_text(encoding="utf-8"))
        except Exception as _read_exc:
            print(f"[warn] Überspringe unlesbare Datei {f.name}: {_read_exc}")
            continue
        _res = _data.get("results") or []
        _by_run = {}
        for _r in _res:
            _rn = int(_r.get("run") or 1)
            _by_run.setdefault(_rn, []).append(_r)
        _uniq_tasks = len({str(_r.get("task") or "") for _r in _res if _r.get("task")})
        print(f"[Overview] model={_data.get('model')} | ctx={_data.get('num_ctx', '?')} | total={len(_res)} cells | unique_tasks={_uniq_tasks}")
        _run_overview = {}
        for _rn in sorted(_by_run):
            _rlist = _by_run[_rn]
            _t_cnt = len({str(_r.get("task") or "") for _r in _rlist if _r.get("task")})
            _c_cnt = len({str(_r.get("condition") or "") for _r in _rlist if _r.get("condition")})
            _p_cnt = sum(1 for _r in _rlist if _r.get("ok"))
            print(f"  Run {_rn}: {len(_rlist)} cells ({_t_cnt} tasks x {_c_cnt} conditions) | {_p_cnt}/{len(_rlist)} PASS")
            _run_overview[str(_rn)] = {"cells": len(_rlist), "tasks": _t_cnt, "conditions": _c_cnt, "passes": _p_cnt}
        _subprocess_lb.run(
            [_py_lb, str(_analyze_lb), "--results", str(f), "--by-model"],
            check=False,
        )
        _json_proc = _subprocess_lb.run(
            [_py_lb, str(_analyze_lb), "--results", str(f), "--by-model", "--json"],
            capture_output=True, text=True, encoding="utf-8", check=False,
        )
        if _json_proc.returncode == 0 and _json_proc.stdout.strip():
            try:
                _combined_export["per_file"][f.name] = {
                    "model": _data.get("model"),
                    "num_ctx": _data.get("num_ctx"),
                    "runs": _run_overview,
                    "summary": _json_lb.loads(_json_proc.stdout),
                }
                _valid_files.append(str(f))
            except ValueError:
                pass
    if _valid_files:
        _all_proc = _subprocess_lb.run(
            [_py_lb, str(_analyze_lb), "--results", *_valid_files, "--by-model", "--json"],
            capture_output=True, text=True, encoding="utf-8", check=False,
        )
        if _all_proc.returncode == 0 and _all_proc.stdout.strip():
            try:
                _combined_export["combined"] = _json_lb.loads(_all_proc.stdout)
            except ValueError:
                pass
        _summary_out = DRIVE_RESULTS / "_leaderboard_summary.json"
        _summary_out.write_text(_json_lb.dumps(_combined_export, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        print(f"\n[SAVE OK] Zusammenfassung exportiert -> {_summary_out}")

In [ ]:
# 10) Server & VRAM sauber freigeben
!bash /root/rent_bootstrap.sh stop